BusState Processing

In [1]:
# Importing necessary libraries

#%pip install numpy
#%pip install pandas

import numpy as np
import pandas as pd
from pathlib import Path
import zipfile
import os
import re


In [2]:
# Set base directory path - should never need changed unless file system changes
# Note: if ever run on unix, change this to reflect unix file system
root_dir = "K:/AP/TTM/"

data_dir = os.path.join(root_dir, "Data/APC Data") # contains zipped raw busstate txt files
repo_dir = os.path.join(root_dir, "WMC Dashboard/BusState Cleaned") # temp file to store cleaned data - change in future if necessary


In [4]:
# Set year/month 
# MUST be in 2 digit format
year = "25"
month = "10"

# Bustate naming convention is busstate0####DDMMYY.txt -> #### is unique 4 digit bus identifier
# IF this ever changes in the future, change the regex pattern below to reflect new naming convention
pattern = re.compile(rf"{year}{month}\d{{2}}\.txt\.zip$") # ending of files is YYMMDD.txt.zip

#print(os.listdir(data_dir)[:10]) # First 10 files in data directory for debugging

# list comprehension to get list of all busstate files matching the year/month - will be zipped
# takes about 30 seconds to run for 1 month of data
busstates = [
    os.path.normpath(os.path.join(data_dir, f)) 
    for f in os.listdir(data_dir) 
    if pattern.search(f)]


In [6]:
# print(busstates[:10])
# print(sum(1 for _ in busstates)) # Count of .zip files

# # # Debugging
# # print(os.path.exists(data_dir)) # Does the file path exist?
# # print(os.listdir(data_dir)[:10]) # First 10 files

In [7]:
def unzip_busstate_to_df(zip_path):
    '''
    Unzip A SINGLE busstate zip file and return a pandas dataframe
    NOTE: This assumes ONE file in the zip, which is the case for all busstate zip files. If this ever changes, we will need to modify this function to handle multiple files in the zip.
    NOTE: The first row of the csv file contains data types, so we will drop that row and reset the index.

    Parameters:
        zip_path (str): Path to the busstate zip file

    Returns:
        pandas dataframe containing all busstate data from the zip file specified 

    
    '''
    with zipfile.ZipFile(zip_path, 'r') as zip_ref:
        # Key assumption is there is only ONE file in the zip 
        file_name = zip_ref.namelist()[0]
        with zip_ref.open(file_name) as f:
            df = (pd.read_csv(f, sep=',', low_memory=False)) # Read all columns as string to avoid dtype issues - we will convert to correct dtypes later
            df = df.drop(df.index[0]).reset_index(drop=True) # DROP first row with "data types"
    return df





In [8]:
def process_busstate(df):
    '''
    Clean and process the busstate dataframe by:
    - Parsing datetime columns
    - Splitting EVENT_TIME into DATE + TIME
    - Keeping only time portion for other datetime columns
    - Converting numeric columns to numeric dtype
    - Creating BOARDINGS + ALIGHTINGS columns
    - Filtering EVENT_TYPE < 17
    - Selecting relevant columns
    - Filtering missing RUN_ID / DEST_SIGN_ROUTE_TEXT
    - Sorting by DATE + EVENT_TIME

    Parameters:
        df (pandas dataframe): Raw unzipped busstate dataframe

    Returns:
        pandas dataframe: Cleaned and processed busstate dataframe
    '''
    #print("BEFORE: ", df["EVENT_TIME"].head(10))

    # cols in YYMMDDhhmmss format
    datetime_cols = [
        "EVENT_TIME",
        "ENTER_STOP_WINDOW_TIME",
        "EXIT_STOP_WINDOW_TIME",
        "TRIP_START_TIME",
        "DEPARTURE_TIME"
    ]

    # convert to datetime,
    for col in datetime_cols:
        df[col] = pd.to_datetime(df[col], format="%y%m%d%H%M%S", errors="coerce")

    #print("AFTER: ", df["EVENT_TIME"].head(10))

    df["DATE"] = df["EVENT_TIME"].dt.date
    df["EVENT_TIME"] = df["EVENT_TIME"].dt.time

    # For the other datetime columns, we only care about the time portion, so we will keep only the time portion
    for col in datetime_cols[1:]:
        df[col] = df[col].dt.time

    # convert numerics from str
    numeric_cols = [
        "STOP_BACK_DOOR_ENTRY",
        "STOP_FRONT_DOOR_ENTRY",
        "STOP_FRONT_DOOR_EXIT",
        "STOP_BACK_DOOR_EXIT",
        "EVENT_TYPE"
    ]

    for col in numeric_cols:
        df[col] = pd.to_numeric(df[col], errors="coerce")

    # baordings and alightnings
    df["BOARDINGS"] = df["STOP_BACK_DOOR_ENTRY"] + df["STOP_FRONT_DOOR_ENTRY"]
    df["ALIGHTINGS"] = df["STOP_FRONT_DOOR_EXIT"] + df["STOP_BACK_DOOR_EXIT"]

    ##################################
    # not sure why this is BUT is in legacy R code so keeping for now - need to investigate later
    # comment stats this is extraneous event types that we don't care about -  will investigate later if we need to keep any of these event types
    df = df[df["EVENT_TYPE"] < 17]

    df = df[[
        "DATE", "BUS_ID", "RUN_ID", "DEST_SIGN_ROUTE_TEXT", "BLOCK_ID", "TRIP_ID",
        "ROUTE_ID", "STOP_SEQUENCE", "LATITUDE", "LONGITUDE",
        "HEADING", "OPERATOR_ID", "ODOMETER_DISTANCE", "TIMEPOINT_ID",
        "EVENT_TYPE", "EVENT_TIME", "BOARDINGS", "ALIGHTINGS", "PASSENGER_LOAD",
        "TRIP_START_TIME", "DEPARTURE_TIME", "ENTER_STOP_WINDOW_TIME", "EXIT_STOP_WINDOW_TIME"
    ]]

    # filter out missing run id and dest signs 
    df = df[(df["RUN_ID"].notna()) | (df["DEST_SIGN_ROUTE_TEXT"].notna())]

    # arrange based on date and time - also in legacy R code
    df = df.sort_values(by=["DATE", "EVENT_TIME"])

    return df

In [9]:
# # # create empty dataframe to hold all busstate data
# # df = pd.DataFrame()

# # # unzip each busstate file and add to 1 dataframe - should take around 45 seconds for 1 month of data
# # for busstate in busstates:
# #     df = pd.concat([df, unzip_busstate_to_df(busstate)])


# raw_data = unzip_busstate_to_df(busstates[0])
# cleaned_data = process_busstate(raw_data)

# cleaned_data.info()


In [10]:
# empty dataframe to hold busstate data
df = pd.DataFrame()

# loop through each busstate file, unzip, process, and add to main dataframe
for busstate in busstates:
    raw_data = unzip_busstate_to_df(busstate)
    cleaned_data = process_busstate(raw_data)
    df = pd.concat([df, cleaned_data], ignore_index=True)

In [ ]:
df.columns

df.head(25)
#df.info()
# info matches the previous R script's output. in good shape!

,DATE,BUS_ID,RUN_ID,DEST_SIGN_ROUTE_TEXT,BLOCK_ID,TRIP_ID,ROUTE_ID,STOP_SEQUENCE,LATITUDE,LONGITUDE,...,TIMEPOINT_ID,EVENT_TYPE,EVENT_TIME,BOARDINGS,ALIGHTINGS,PASSENGER_LOAD,TRIP_START_TIME,DEPARTURE_TIME,ENTER_STOP_WINDOW_TIME,EXIT_STOP_WINDOW_TIME
0,2025-09-30,1301,NaN,H008,NaN,NaN,NaN,-1,40.002136,-83.037033,...,NaN,14,13:56:45,0,0,0,NaT,NaT,NaT,NaT
1,2025-09-30,1301,NaN,H008,23675202,NaN,NaN,-1,40.002136,-83.037033,...,NaN,14,13:56:46,0,0,0,NaT,NaT,NaT,NaT
2,2025-09-30,1301,1502,H008,23675202,NaN,NaN,-1,40.002136,-83.037033,...,NaN,7,13:56:47,0,0,0,NaT,NaT,NaT,NaT
3,2025-09-30,1301,1502,MC,23675202,NaN,MC05,-1,40.002258,-83.038658,...,NaN,6,13:56:48,0,0,0,NaT,NaT,NaT,NaT
4,2025-09-30,1301,1502,MC,23675202,1622020,MC05,-1,40.002258,-83.038658,...,NaN,15,13:56:48,0,0,0,13:56:47,NaT,NaT,NaT
5,2025-09-30,1301,1502,MC,23675202,1622020,MC05,-1,40.002258,-83.038658,...,NaN,9,13:56:48,0,0,0,13:56:47,NaT,NaT,NaT
6,2025-09-30,1301,1502,MC,23675202,1622020,MC05,-1,40.002258,-83.038658,...,NaN,10,13:56:48,0,0,0,13:56:47,NaT,NaT,NaT
7,2025-09-30,1301,1502,MC,23675202,1622020,MC05,-1,40.002258,-83.038658,...,NaN,8,13:56:51,0,0,0,13:56:47,NaT,NaT,NaT
8,2025-09-30,1301,1502,MC,23675202,1622020,MC05,1,40.000179,-83.037201,...,CMCK2,16,13:57:53,0,0,0,13:56:47,13:59:41,NaT,NaT
9,2025-09-30,1301,1502,MC,23675202,1622020,MC05,1,40.001087,-83.037086,...,NaN,3,13:58:10,1,2,0,13:56:47,13:59:10,13:57:57,13:59:23
